# Module 11 — Reserve Risk Early Warning System

**Objectif** : détecter si les facteurs de développement (LDFs) du triangle CAS dévient
significativement dans le temps — signal précoce d'une **social inflation** ou d'un
changement de comportement de sinistres avant que Chain Ladder ne sous-estime les réserves.

**Données** : CAS Loss Reserve Database — *New Jersey Manufacturers Grp*, LOB `ppauto`
(équivalent GRCODE 7080) — validé au Module 3 (réserve STD ≈ 494 113 $, MAPE ≈ 1.24%).

**Méthodes** :
- Test de Welch bilatéral par lag (alerte FORTE p<0.05, MODÉRÉE p<0.10)
- Tendance linéaire OLS (LDF ~ AccidentYear) par lag
- Impact réserves : 3 scénarios CL (toutes années / historique / récent)
- Forecast LDFs à +2 ans par extrapolation OLS
- Score de risque global (0–100) + classification VERT / AMBRE / ROUGE

**Split récent/historique** : adaptatif par lag — N_RECENT=3 dernières AYs
observant ce lag (réduit à min(N_RECENT, n_av−2) si données insuffisantes).

In [1]:
# ============================================================
# BLOC 0 — Installation des dépendances
# ============================================================
!pip install chainladder scipy statsmodels --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [2]:
# ============================================================
# BLOC 1 — Chargement des données CAS
# ============================================================
import chainladder as cl
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# --- Paramètres ---
N_RECENT      = 3
GRNAME_TARGET = 'New Jersey Manufacturers Grp'
LOB_TARGET    = 'ppauto'
ALPHA_FORT    = 0.05
ALPHA_MODERE  = 0.10
COL_PAID      = 'CumPaidLoss'
COL_PREM      = 'EarnedPremNet'

# --- Chargement ---
clrd    = cl.load_sample('clrd')
df_flat = clrd.to_frame(origin_as_datetime=False).reset_index()

print('Colonnes brutes :', df_flat.columns.tolist())
print('GRNAMEs NJ disponibles :',
      [g for g in df_flat['GRNAME'].unique() if 'Jersey' in g or 'Manufact' in g])

# --- Filtrage société + LOB ---
mask = (df_flat['GRNAME'] == GRNAME_TARGET) & (df_flat['LOB'] == LOB_TARGET)
df_company = df_flat[mask].copy()
assert len(df_company) > 0, 'Filtre vide — vérifier GRNAME_TARGET et LOB_TARGET'

# --- Conversion des types ---
# 'origin' est un pandas Period (ex. Period('1988','Y-DEC')) -> année entière
# 'development' est en mois (12, 24, ...) -> lag annuel (1, 2, ...)
df_company['AccidentYear']   = df_company['origin'].apply(lambda p: p.year)
df_company['DevelopmentLag'] = (df_company['development'] // 12).astype(int)

print(f'\nLignes chargées     : {len(df_company)}')
print(f'AccidentYears       : {sorted(df_company["AccidentYear"].unique())}')
print(f'DevelopmentLags     : {sorted(df_company["DevelopmentLag"].unique())}')
print(f'Colonne sinistres   : {COL_PAID}')
print(f'Colonne primes      : {COL_PREM}')

Colonnes brutes : ['GRNAME', 'LOB', 'origin', 'development', 'IncurLoss', 'CumPaidLoss', 'BulkLoss', 'EarnedPremDIR', 'EarnedPremCeded', 'EarnedPremNet']
GRNAMEs NJ disponibles : ['New Jersey Citizens United Rcp Exch', 'New Jersey Manufacturers Grp']

Lignes chargées     : 55
AccidentYears       : [np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997)]
DevelopmentLags     : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Colonne sinistres   : CumPaidLoss
Colonne primes      : EarnedPremNet


In [3]:
# ============================================================
# BLOC 2 — Construction du triangle + matrice LDF_ij
# ============================================================
n_dev  = df_company['DevelopmentLag'].nunique()   # 10
max_ay = int(df_company['AccidentYear'].max())    # 1997

# Triangle upper-left : AY + lag - 1 <= max_ay
df_triangle = df_company[
    df_company['AccidentYear'] + df_company['DevelopmentLag'] - 1 <= max_ay
].copy()

print(f'Lignes triangle (attendu 55) : {len(df_triangle)}')
assert len(df_triangle) == 55, 'Triangle incorrect'

# --- Pivot pour calcul des LDFs ---
origin_years = [int(y) for y in sorted(df_triangle['AccidentYear'].unique())]
paid_pivot   = df_triangle.pivot(index='AccidentYear', columns='DevelopmentLag', values=COL_PAID)
paid_pivot.index   = paid_pivot.index.astype(int)
paid_pivot.columns = paid_pivot.columns.astype(int)

# --- Matrice LDF_ij : LDF(AY=i, lag=k->k+1) = C(i,k+1) / C(i,k) ---
ldf_matrix = {}
for lag in range(1, n_dev):
    d = {}
    for ay in origin_years:
        cc = paid_pivot.loc[ay, lag]   if lag   in paid_pivot.columns else np.nan
        cn = paid_pivot.loc[ay, lag+1] if lag+1 in paid_pivot.columns else np.nan
        if pd.notna(cc) and pd.notna(cn) and cc > 0:
            d[ay] = float(cn / cc)
    ldf_matrix[lag] = d

# Affichage
ldf_df = pd.DataFrame(ldf_matrix).rename(columns=lambda k: f'LDF_{k}->{k+1}')
print('\nMatrice LDF par année de survenance :')
print(ldf_df.round(4).to_string())
print(f'\nn_dev={n_dev}, max_ay={max_ay}, lags calculés={len(ldf_matrix)}')

Lignes triangle (attendu 55) : 55

Matrice LDF par année de survenance :
      LDF_1->2  LDF_2->3  LDF_3->4  LDF_4->5  LDF_5->6  LDF_6->7  LDF_7->8  LDF_8->9  LDF_9->10
1988    1.9893    1.3506    1.2208    1.1492    1.0584    1.0295    1.0103    1.0201     1.0031
1989    1.9405    1.3668    1.2279    1.1516    1.0946    1.0336    1.0155    1.0129        NaN
1990    1.9326    1.3745    1.3153    1.1919    1.1066    1.0388    1.0130       NaN        NaN
1991    1.9573    1.4286    1.3649    1.2619    1.0907    1.0280       NaN       NaN        NaN
1992    1.9467    1.4442    1.3596    1.2450    1.0735       NaN       NaN       NaN        NaN
1993    2.0185    1.4351    1.3920    1.2218       NaN       NaN       NaN       NaN        NaN
1994    2.0235    1.4625    1.3290       NaN       NaN       NaN       NaN       NaN        NaN
1995    2.1648    1.4380       NaN       NaN       NaN       NaN       NaN       NaN        NaN
1996    2.0007       NaN       NaN       NaN       NaN       Na

In [4]:
# ============================================================
# BLOC 3 — Split récent/historique + statistiques descriptives
#
# Split ADAPTATIF par lag :
#   n_recent = min(N_RECENT, n_disponible - 2)  -> garantit >= 2 hist et >= 1 recent
#   Les lags avec < 3 AY sont marqués 'insufficient' (inclus pour complétude)
# ============================================================
stats = []
for lag, ay_dict in ldf_matrix.items():
    avail = sorted(ay_dict.keys())
    n_av  = len(avail)

    if n_av < 3:
        stats.append({
            'lag': f'{lag}->{lag+1}', 'lag_num': lag,
            'n_hist': 0, 'n_recent': n_av,
            'mean_hist': None, 'mean_recent': float(np.mean(list(ay_dict.values()))),
            'std_hist': None, 'std_recent': 0.0, 'ecart_pct': None,
            'hist_vals': [], 'recent_vals': list(ay_dict.values()),
            'all_vals': list(ay_dict.values()), 'all_years': avail,
            'insufficient': True,
        })
        continue

    n_rec      = min(N_RECENT, n_av - 2)
    recent_ays = avail[-n_rec:]
    hist_ays   = avail[:-n_rec]
    hv = [ay_dict[a] for a in hist_ays]
    rv = [ay_dict[a] for a in recent_ays]

    stats.append({
        'lag': f'{lag}->{lag+1}', 'lag_num': lag,
        'n_hist': len(hv), 'n_recent': len(rv),
        'mean_hist': float(np.mean(hv)), 'mean_recent': float(np.mean(rv)),
        'std_hist': float(np.std(hv, ddof=1)) if len(hv) > 1 else 0.0,
        'std_recent': float(np.std(rv, ddof=1)) if len(rv) > 1 else 0.0,
        'ecart_pct': float((np.mean(rv) - np.mean(hv)) / np.mean(hv) * 100),
        'hist_vals': hv, 'recent_vals': rv,
        'all_vals': list(ay_dict.values()), 'all_years': avail,
        'insufficient': False,
    })

stats_df = pd.DataFrame(
    [{k: v for k, v in s.items()
      if k not in ['hist_vals','recent_vals','all_vals','all_years','insufficient']}
     for s in stats]
)
print('Statistiques descriptives par lag (split adaptatif) :')
print(stats_df[['lag','n_hist','n_recent','mean_hist','mean_recent','ecart_pct']]
      .round(4).to_string(index=False))

Statistiques descriptives par lag (split adaptatif) :
  lag  n_hist  n_recent  mean_hist  mean_recent  ecart_pct
 1->2       6         3     1.9642       2.0630     5.0310
 2->3       5         3     1.3929       1.4452     3.7551
 3->4       4         3     1.2822       1.3602     6.0816
 4->5       3         3     1.1642       1.2429     6.7555
 5->6       2         3     1.0765       1.0903     1.2775
 6->7       2         2     1.0316       1.0334     0.1804
 7->8       2         1     1.0129       1.0130     0.0173
 8->9       0         2        NaN       1.0165        NaN
9->10       0         1        NaN       1.0031        NaN


In [5]:
# ============================================================
# BLOC 4 — Test de Welch bilatéral par lag
# ============================================================
from scipy import stats as scipy_stats

ttest_results = []
for s in stats:
    if s.get('insufficient') or s['n_hist'] < 2:
        ttest_results.append({
            **s, 't_stat': None, 'p_value': None,
            'alert_level': 'DONNÉES INSUFFISANTES', 'ttest_note': 'n_total < 3'
        })
        continue

    hv, rv = s['hist_vals'], s['recent_vals']
    if len(rv) < 2:
        t_stat, p_val = np.nan, 1.0
        note = f'n_recent={len(rv)} — p non calculable, fallback écart %'
    else:
        t_stat, p_val = scipy_stats.ttest_ind(rv, hv, equal_var=False)
        note = ''

    alert = 'FORTE' if p_val < ALPHA_FORT else 'MODÉRÉE' if p_val < ALPHA_MODERE else 'AUCUNE'
    if len(rv) < 2 and s['ecart_pct'] is not None and abs(s['ecart_pct']) > 15:
        alert = 'MODÉRÉE'
        note += f' | écart {s["ecart_pct"]:.1f}% > 15%'

    ttest_results.append({
        **s,
        't_stat': round(float(t_stat), 4) if not np.isnan(t_stat) else None,
        'p_value': round(float(p_val), 4),
        'alert_level': alert,
        'ttest_note': note,
    })

# Affichage
tt_df = pd.DataFrame(
    [{k: v for k, v in r.items()
      if k not in ['hist_vals','recent_vals','all_vals','all_years','ttest_note','insufficient']}
     for r in ttest_results]
)
print('Test de Welch par lag :')
print(tt_df[['lag','mean_hist','mean_recent','ecart_pct','t_stat','p_value','alert_level']]
      .round(4).to_string(index=False))

n_forte  = sum(1 for r in ttest_results if r['alert_level'] == 'FORTE')
n_modere = sum(1 for r in ttest_results if r['alert_level'] == 'MODÉRÉE')
print(f'\nAlertes FORTES   (p<0.05) : {n_forte}')
print(f'Alertes MODÉRÉES (p<0.10) : {n_modere}')

Test de Welch par lag :
  lag  mean_hist  mean_recent  ecart_pct  t_stat  p_value           alert_level
 1->2     1.9642       2.0630     5.0310  1.8621   0.1877                AUCUNE
 2->3     1.3929       1.4452     3.7551  2.5791   0.0456                 FORTE
 3->4     1.2822       1.3602     6.0816  1.9795   0.1130                AUCUNE
 4->5     1.1642       1.2429     6.7555  4.3462   0.0130                 FORTE
 5->6     1.0765       1.0903     1.2775  0.6713   0.5867                AUCUNE
 6->7     1.0316       1.0334     0.1804  0.3236   0.7909                AUCUNE
 7->8     1.0129       1.0130     0.0173     NaN   1.0000                AUCUNE
 8->9        NaN       1.0165        NaN     NaN      NaN DONNÉES INSUFFISANTES
9->10        NaN       1.0031        NaN     NaN      NaN DONNÉES INSUFFISANTES

Alertes FORTES   (p<0.05) : 2
Alertes MODÉRÉES (p<0.10) : 0


In [6]:
# ============================================================
# BLOC 5 — Tendance linéaire OLS (LDF ~ AccidentYear) par lag
# ============================================================
from scipy.stats import linregress

trend_results = []
for r in ttest_results:
    years, vals = r['all_years'], r['all_vals']
    if len(years) < 3:
        trend_results.append({**r, 'slope': None, 'r_squared': None,
                               'trend_pvalue': None, 'trend_dir': 'INDÉTERMINÉ',
                               'trend_signif': False, 'intercept': None})
        continue

    slope, intercept, r_val, p_ols, _ = linregress(years, vals)
    r2 = r_val ** 2
    td = ('HAUSSE' if p_ols < 0.10 and slope > 0
          else 'BAISSE' if p_ols < 0.10 and slope < 0
          else 'STABLE')
    trend_results.append({
        **r,
        'slope'       : round(float(slope), 6),
        'r_squared'   : round(float(r2), 4),
        'trend_pvalue': round(float(p_ols), 4),
        'intercept'   : float(intercept),
        'trend_dir'   : td,
        'trend_signif': bool(p_ols < 0.10),
    })

tr_df = pd.DataFrame(
    [{k: v for k, v in r.items()
      if k not in ['hist_vals','recent_vals','all_vals','all_years','ttest_note','intercept','insufficient']}
     for r in trend_results]
)
print('Tendance OLS par lag :')
print(tr_df[['lag','slope','r_squared','trend_pvalue','trend_dir','alert_level']]
      .round(4).to_string(index=False))

Tendance OLS par lag :
  lag  slope  r_squared  trend_pvalue   trend_dir           alert_level
 1->2 0.0160     0.3773        0.0784      HAUSSE                AUCUNE
 2->3 0.0153     0.8034        0.0026      HAUSSE                 FORTE
 3->4 0.0249     0.6414        0.0304      HAUSSE                AUCUNE
 4->5 0.0204     0.6459        0.0540      HAUSSE                 FORTE
 5->6 0.0026     0.0478        0.7240      STABLE                AUCUNE
 6->7 0.0001     0.0005        0.9782      STABLE                AUCUNE
 7->8 0.0014     0.2844        0.6419      STABLE                AUCUNE
 8->9    NaN        NaN           NaN INDÉTERMINÉ DONNÉES INSUFFISANTES
9->10    NaN        NaN           NaN INDÉTERMINÉ DONNÉES INSUFFISANTES


In [9]:
# ============================================================
# BLOC 6 CORRIGÉ — 3 scénarios CL avec fallback pour RÉCENT
# ============================================================
all_ay_global       = [int(y) for y in sorted(origin_years)]
hist_years_global   = all_ay_global[:-N_RECENT]   # [1988..1994]
recent_years_global = all_ay_global[-N_RECENT:]   # [1995, 1996, 1997]

def compute_cl_reserve_v2(df_tri, paid_col, years_subset=None, fallback_years=None):
    """
    Réserve CL avec LDFs pondérés sur years_subset.
    Si un lag n'a aucune AY dans years_subset, utilise fallback_years
    (ou toutes les années si fallback_years=None).
    """
    p = df_tri.pivot(index='AccidentYear', columns='DevelopmentLag', values=paid_col)
    p.index   = p.index.astype(int)
    p.columns = p.columns.astype(int)
    n = p.shape[1]

    ldfs = []
    sources = []   # pour debug
    for lag in range(1, n):
        def get_rows(subset):
            return [ay for ay in p.index
                    if (subset is None or ay in subset)
                    and lag in p.columns and lag+1 in p.columns
                    and pd.notna(p.loc[ay, lag]) and pd.notna(p.loc[ay, lag+1])]

        rows = get_rows(years_subset) if years_subset else get_rows(None)

        if not rows and fallback_years is not None:
            rows = get_rows(fallback_years)
            sources.append('FALLBACK')
        elif not rows:
            rows = get_rows(None)
            sources.append('ALL')
        else:
            sources.append('SUBSET')

        if not rows:
            ldfs.append(1.0)
            sources[-1] = 'DEFAULT_1.0'
            continue

        num = sum(p.loc[ay, lag+1] for ay in rows)
        den = sum(p.loc[ay, lag]   for ay in rows)
        ldfs.append(num / den if den > 0 else 1.0)

    cdfs = [1.0] * n
    for i in range(n-2, -1, -1):
        cdfs[i] = cdfs[i+1] * ldfs[i]

    reserve = 0
    for ay in p.index:
        obs = [(l, p.loc[ay, l]) for l in p.columns if pd.notna(p.loc[ay, l])]
        if not obs:
            continue
        ll, lv = max(obs, key=lambda x: x[0])
        if ll >= n:
            continue
        reserve += lv * cdfs[ll-1] - lv

    return round(reserve, 0), ldfs, sources

reserve_std,  ldfs_std,  src_std  = compute_cl_reserve_v2(df_triangle, COL_PAID)
reserve_hist, ldfs_hist, src_hist = compute_cl_reserve_v2(
    df_triangle, COL_PAID,
    years_subset=hist_years_global
)
# RÉCENT hybride : LDFs récents si données dispo, STD sinon
reserve_rec,  ldfs_rec,  src_rec  = compute_cl_reserve_v2(
    df_triangle, COL_PAID,
    years_subset=recent_years_global,
    fallback_years=None   # fallback = toutes années (STD)
)

# Debug sources
print("Sources LDF par scénario et lag :")
print(f"{'Lag':<8} {'STD':<10} {'HIST':<10} {'REC (hybride)':<15} {'LDF_REC'}")
for i, lag in enumerate(range(1, 10)):
    print(f"  {lag}->{lag+1:<6} {src_std[i]:<10} {src_hist[i]:<10} "
          f"{src_rec[i]:<15} {ldfs_rec[i]:.4f}")

print(f"\nComparaison des réserves CL :")
print(f"  Scénario STD           : {reserve_std:>12,.0f} $   [référence]")
print(f"  Scénario HIST          : {reserve_hist:>12,.0f} $   "
      f"({(reserve_hist-reserve_std)/reserve_std*100:+.1f}% vs STD)")
print(f"  Scénario RÉCENT hybride: {reserve_rec:>12,.0f} $   "
      f"({(reserve_rec-reserve_std)/reserve_std*100:+.1f}% vs STD)")
print(f"  Plage incertitude      : {abs(reserve_rec-reserve_hist):>12,.0f} $ "
      f"({abs(reserve_rec-reserve_hist)/reserve_std*100:.1f}% des réserves)")

assert abs(reserve_std - 494112) < 5000, f"Réserve STD hors tolérance : {reserve_std}"
print(f"\n Cohérence Module 3 : {reserve_std:,.0f}$ ≈ 494 112$")

# Note méthodologique importante
print("""
NOTE MÉTHODOLOGIQUE :
  Le scénario RÉCENT utilise les LDFs des 3 dernières années (1995-1997)
  uniquement pour les lags où ces années sont observées (lags 1→2 et 2→3).
  Pour les lags 3→10 (non observables sur 1995-1997 car triangle incomplet),
  les LDFs du scénario STD sont utilisés en fallback.
  → Cette approche donne une estimation de l'impact si les tendances
    récentes (lags courts) se propagent aux lags longs.
""")

Sources LDF par scénario et lag :
Lag      STD        HIST       REC (hybride)   LDF_REC
  1->2      SUBSET     SUBSET     SUBSET          2.0734
  2->3      SUBSET     SUBSET     SUBSET          1.4380
  3->4      SUBSET     SUBSET     ALL             1.3224
  4->5      SUBSET     SUBSET     ALL             1.2088
  5->6      SUBSET     SUBSET     ALL             1.0849
  6->7      SUBSET     SUBSET     ALL             1.0324
  7->8      SUBSET     SUBSET     ALL             1.0130
  8->9      SUBSET     SUBSET     ALL             1.0162
  9->10     SUBSET     SUBSET     ALL             1.0031

Comparaison des réserves CL :
  Scénario STD           :      494,113 $   [référence]
  Scénario HIST          :      489,060 $   (-1.0% vs STD)
  Scénario RÉCENT hybride:      508,724 $   (+3.0% vs STD)
  Plage incertitude      :       19,664 $ (4.0% des réserves)

 Cohérence Module 3 : 494,113$ ≈ 494 112$

NOTE MÉTHODOLOGIQUE :
  Le scénario RÉCENT utilise les LDFs des 3 dernières années (199

In [10]:
# BLOC 6bis — Diagnostic du scénario RÉCENT
print("LDFs scénario RÉCENT (debug) :")
_, ldfs_rec_debug = compute_cl_reserve(df_triangle, COL_PAID,
                                        years_subset=recent_years_global)
_, ldfs_hist_debug = compute_cl_reserve(df_triangle, COL_PAID,
                                         years_subset=hist_years_global)
_, ldfs_std_debug = compute_cl_reserve(df_triangle, COL_PAID)

print(f"{'Lag':<8} {'LDF_STD':<12} {'LDF_HIST':<12} {'LDF_REC':<12} {'Rows_REC'}")
for i, lag in enumerate(range(1, 10)):
    rows_rec = [ay for ay in paid_pivot.index.astype(int)
                if ay in recent_years_global
                and lag in paid_pivot.columns and lag+1 in paid_pivot.columns
                and pd.notna(paid_pivot.loc[ay, lag])
                and pd.notna(paid_pivot.loc[ay, lag+1])]
    print(f"  {lag}->{lag+1:<6} {ldfs_std_debug[i]:.4f}      "
          f"{ldfs_hist_debug[i]:.4f}      "
          f"{ldfs_rec_debug[i]:.4f}      "
          f"{len(rows_rec)} AYs")

LDFs scénario RÉCENT (debug) :
Lag      LDF_STD      LDF_HIST     LDF_REC      Rows_REC
  1->2      2.0064      1.9762      2.0734      2 AYs
  2->3      1.4184      1.4141      1.4380      1 AYs
  3->4      1.3224      1.3224      1.0000      0 AYs
  4->5      1.2088      1.2088      1.0000      0 AYs
  5->6      1.0849      1.0849      1.0000      0 AYs
  6->7      1.0324      1.0324      1.0000      0 AYs
  7->8      1.0130      1.0130      1.0000      0 AYs
  8->9      1.0162      1.0162      1.0000      0 AYs
  9->10     1.0031      1.0031      1.0000      0 AYs


In [11]:
# ============================================================
# BLOC 7 — Forecast LDFs à +2 ans (extrapolation OLS)
# Zone de danger : forecast > mean_hist * 1.15
# ============================================================
forecast_results = []
for r in trend_results:
    if r['slope'] is None:
        forecast_results.append({
            **r, 'forecast_1998': None, 'forecast_1999': None,
            'forecast_danger': False, 'danger_threshold': None
        })
        continue

    f98 = round(r['slope'] * 1998 + r['intercept'], 4)
    f99 = round(r['slope'] * 1999 + r['intercept'], 4)
    mh  = r['mean_hist'] if r['mean_hist'] is not None else r['mean_recent']
    thr = round(mh * 1.15, 4)

    forecast_results.append({
        **r,
        'forecast_1998'   : f98,
        'forecast_1999'   : f99,
        'forecast_danger' : bool(f98 > thr or f99 > thr),
        'danger_threshold': thr,
    })

fc_df = pd.DataFrame(
    [{k: v for k, v in r.items()
      if k not in ['hist_vals','recent_vals','all_vals','all_years','ttest_note','intercept','insufficient']}
     for r in forecast_results]
)
print('Forecast LDFs (OLS) :')
print(fc_df[['lag','mean_hist','mean_recent','forecast_1998','forecast_1999',
             'danger_threshold','forecast_danger','trend_dir']]
      .round(4).to_string(index=False))

n_danger = sum(1 for r in forecast_results if r.get('forecast_danger'))
print(f'\nLags en zone de danger (forecast > hist x 1.15) : {n_danger}')

Forecast LDFs (OLS) :
  lag  mean_hist  mean_recent  forecast_1998  forecast_1999  danger_threshold  forecast_danger   trend_dir
 1->2     1.9642       2.0630         2.0933         2.1094            2.2588            False      HAUSSE
 2->3     1.3929       1.4452         1.5116         1.5269            1.6019            False      HAUSSE
 3->4     1.2822       1.3602         1.4902         1.5151            1.4746             True      HAUSSE
 4->5     1.1642       1.2429         1.3572         1.3775            1.3389             True      HAUSSE
 5->6     1.0765       1.0903         1.1066         1.1092            1.2380            False      STABLE
 6->7     1.0316       1.0334         1.0341         1.0342            1.1863            False      STABLE
 7->8     1.0129       1.0130         1.0246         1.0260            1.1648            False      STABLE
 8->9        NaN       1.0165            NaN            NaN               NaN            False INDÉTERMINÉ
9->10        Na

In [12]:
# ============================================================
# BLOC 8 — Score de risque global (0–100) + classification
# Seuls les lags avec données suffisantes entrent dans le score
# ============================================================
valid = [r for r in forecast_results if r['alert_level'] != 'DONNÉES INSUFFISANTES']
n_lt  = len(valid)

n_alert_forte  = sum(1 for r in valid if r['alert_level'] == 'FORTE')
n_alert_modere = sum(1 for r in valid if r['alert_level'] == 'MODÉRÉE')
n_trend_hausse = sum(1 for r in valid if r['trend_dir'] == 'HAUSSE')
n_forecast_dng = sum(1 for r in valid if r.get('forecast_danger'))
max_ecart_abs  = max(
    (abs(r['ecart_pct']) for r in valid if r['ecart_pct'] is not None), default=0
)

score_ttest    = (n_alert_forte * 15 + n_alert_modere * 8) / max(n_lt, 1)
score_trend    = n_trend_hausse / max(n_lt, 1) * 25
score_forecast = n_forecast_dng / max(n_lt, 1) * 30
score_ecart    = min(max_ecart_abs / 50, 1) * 20
raw_score      = score_ttest + score_trend + score_forecast + score_ecart
normalizer     = (15 * max(n_lt, 1) + 25 + 30 + 20) / max(n_lt, 1)
global_score   = min(round(raw_score / normalizer * 100, 1), 100)

if global_score < 25:
    classification = 'VERT'
    label = 'Risque faible — aucune action requise'
elif global_score < 55:
    classification = 'AMBRE'
    label = 'Risque modéré — surveillance renforcée'
else:
    classification = 'ROUGE'
    label = 'Risque élevé — révision des réserves recommandée'

# Alertes textuelles
alerts = []
for r in forecast_results:
    if r['alert_level'] == 'FORTE':
        alerts.append({'type': 'FORTE', 'lag': r['lag'],
            'message': f"Lag {r['lag']} : déviation significative (p={r['p_value']:.3f}<0.05), écart {r['ecart_pct']:+.1f}%"})
    elif r['alert_level'] == 'MODÉRÉE':
        alerts.append({'type': 'MODÉRÉE', 'lag': r['lag'],
            'message': f"Lag {r['lag']} : déviation modérée (p={r['p_value']:.3f}<0.10), écart {r['ecart_pct']:+.1f}%"})
    if r.get('forecast_danger') and r.get('forecast_1999') is not None:
        alerts.append({'type': 'FORECAST', 'lag': r['lag'],
            'message': f"Lag {r['lag']} : forecast 1999 ({r['forecast_1999']:.4f}) > seuil ({r['danger_threshold']:.4f})"})

print('=' * 60)
print('SCORE DE RISQUE GLOBAL — RESERVE RISK EARLY WARNING')
print('=' * 60)
print(f'Score          : {global_score} / 100')
print(f'Classification : {classification}')
print(f'Verdict        : {label}')
print(f'\nDétail (sur {n_lt} lags valides) :')
print(f'  Alertes t-test FORTES   : {n_alert_forte}')
print(f'  Alertes t-test MODÉRÉES : {n_alert_modere}')
print(f'  Tendances en hausse     : {n_trend_hausse}')
print(f'  Lags en zone danger     : {n_forecast_dng}')
print(f'  Écart récent/hist max   : {max_ecart_abs:.1f}%')
print(f'\nImpact réserves :')
print(f'  Réserve STD      : {reserve_std:>12,.0f} $')
print(f'  Réserve HIST     : {reserve_hist:>12,.0f} $ ({(reserve_hist-reserve_std)/reserve_std*100:+.1f}%)')
print(f'  Réserve RÉCENT   : {reserve_rec:>12,.0f} $ ({(reserve_rec-reserve_std)/reserve_std*100:+.1f}%)')
print(f'  Plage incertitude: {abs(reserve_rec-reserve_hist):>12,.0f} $ ({abs(reserve_rec-reserve_hist)/reserve_std*100:.1f}% des réserves)')
print(f'\nAlertes ({len(alerts)}) :')
for a in alerts:
    pfx = '[ROUGE]' if a['type']=='FORTE' else '[AMBRE]' if a['type']=='MODÉRÉE' else '[TREND]'
    print(f'  {pfx} {a["message"]}')

SCORE DE RISQUE GLOBAL — RESERVE RISK EARLY WARNING
Score          : 100 / 100
Classification : ROUGE
Verdict        : Risque élevé — révision des réserves recommandée

Détail (sur 7 lags valides) :
  Alertes t-test FORTES   : 2
  Alertes t-test MODÉRÉES : 0
  Tendances en hausse     : 4
  Lags en zone danger     : 2
  Écart récent/hist max   : 6.8%

Impact réserves :
  Réserve STD      :      494,113 $
  Réserve HIST     :      489,060 $ (-1.0%)
  Réserve RÉCENT   :      508,724 $ (+3.0%)
  Plage incertitude:       19,664 $ (4.0% des réserves)

Alertes (4) :
  [ROUGE] Lag 2->3 : déviation significative (p=0.046<0.05), écart +3.8%
  [TREND] Lag 3->4 : forecast 1999 (1.5151) > seuil (1.4746)
  [ROUGE] Lag 4->5 : déviation significative (p=0.013<0.05), écart +6.8%
  [TREND] Lag 4->5 : forecast 1999 (1.3775) > seuil (1.3389)


In [13]:
# ============================================================
# BLOC 9 — Checklist de validation
# ============================================================
print('CHECKLIST DE VALIDATION — MODULE 11')
print('-' * 50)

checks = [
    ('Triangle 55 lignes',                len(df_triangle) == 55),
    ('Matrice LDF : 9 lags',              len(ldf_matrix) == 9),
    ('Stats calculées (>= 5 lags)',        len(stats) >= 5),
    ('Tests Welch calculés',              len(ttest_results) == 9),
    ('Tendances OLS calculées',           len(trend_results) == 9),
    ('Réserve STD ~ 494 112 $',           abs(reserve_std - 494112) < 5000),
    ('3 scénarios distincts',             reserve_hist != reserve_std and reserve_rec != reserve_std),
    ('Forecast 1998-1999 produit',        any(r.get('forecast_1998') for r in forecast_results)),
    ('Score 0-100',                       0 <= global_score <= 100),
    ('Classification valide',            classification in ['VERT','AMBRE','ROUGE']),
]

all_ok = True
for lbl_c, ok in checks:
    print(f"  {'OK' if ok else 'ERREUR':6s} {lbl_c}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print(f'TOUTES LES VALIDATIONS PASSÉES')
    print(f'Score : {global_score}/100 — {classification} — {label}')
else:
    print('Certaines validations ont échoué — corriger avant export')

CHECKLIST DE VALIDATION — MODULE 11
--------------------------------------------------
  OK     Triangle 55 lignes
  OK     Matrice LDF : 9 lags
  OK     Stats calculées (>= 5 lags)
  OK     Tests Welch calculés
  OK     Tendances OLS calculées
  OK     Réserve STD ~ 494 112 $
  OK     3 scénarios distincts
  OK     Forecast 1998-1999 produit
  OK     Score 0-100
  OK     Classification valide

TOUTES LES VALIDATIONS PASSÉES
Score : 100/100 — ROUGE — Risque élevé — révision des réserves recommandée


In [14]:
# ============================================================
# BLOC 10 — Export JSON artifact
# ============================================================
import os, shutil
from datetime import datetime

os.makedirs('ew_artifacts', exist_ok=True)

def clean(r):
    skip = {'hist_vals','recent_vals','all_vals','all_years','ttest_note','intercept','insufficient'}
    return {
        k: (float(v) if isinstance(v, (np.floating, np.integer)) else
            bool(v)  if isinstance(v, (np.bool_,)) else v)
        for k, v in r.items() if k not in skip
    }

ldf_series = {}
for r in forecast_results:
    years_r = r.get('all_years') or []
    vals_r  = r.get('all_vals')  or []
    hist_ays_set   = set(years_r[:-min(N_RECENT, len(years_r)-2)]) if len(years_r)>=3 else set()
    recent_ays_set = set(years_r[-min(N_RECENT, len(years_r)-2):]) if len(years_r)>=3 else set(years_r)
    ldf_series[r['lag']] = {
        'historical'      : [{'year': int(y), 'ldf': round(v,4)} for y,v in zip(years_r,vals_r) if y in hist_ays_set],
        'recent'          : [{'year': int(y), 'ldf': round(v,4)} for y,v in zip(years_r,vals_r) if y in recent_ays_set],
        'forecast'        : [{'year':1998,'ldf':r.get('forecast_1998')},{'year':1999,'ldf':r.get('forecast_1999')}],
        'mean_hist'       : r['mean_hist'],
        'mean_recent'     : r['mean_recent'],
        'danger_threshold': r.get('danger_threshold'),
    }

report = {
    'generated_at'  : datetime.now().isoformat(),
    'source'        : f'CAS Loss Reserve Database — {GRNAME_TARGET} / {LOB_TARGET}',
    'methodology'   : {
        'n_recent_years'             : N_RECENT,
        'split'                      : 'adaptatif par lag',
        'alpha_fort'                 : ALPHA_FORT,
        'alpha_modere'               : ALPHA_MODERE,
        'forecast_danger_multiplier' : 1.15,
        'global_ay_hist'             : all_ay_global[:-N_RECENT],
        'global_ay_recent'           : all_ay_global[-N_RECENT:],
    },
    'global_score'  : global_score,
    'classification': classification,
    'verdict'       : label,
    'alerts'        : alerts,
    'summary'       : {
        'n_lags_total'      : len(forecast_results),
        'n_lags_valid'      : n_lt,
        'n_alerts_forte'    : n_alert_forte,
        'n_alerts_modere'   : n_alert_modere,
        'n_trends_hausse'   : n_trend_hausse,
        'n_forecast_danger' : n_forecast_dng,
        'max_ecart_pct'     : round(max_ecart_abs, 2),
    },
    'reserves'      : {
        'standard'               : float(reserve_std),
        'historical_ldfs'        : float(reserve_hist),
        'recent_ldfs'            : float(reserve_rec),
        'delta_hist_pct'         : round((reserve_hist-reserve_std)/reserve_std*100, 2),
        'delta_recent_pct'       : round((reserve_rec-reserve_std)/reserve_std*100, 2),
        'uncertainty_range'      : float(abs(reserve_rec-reserve_hist)),
        'uncertainty_pct'        : round(abs(reserve_rec-reserve_hist)/reserve_std*100, 2),
    },
    'lag_details'   : [clean(r) for r in forecast_results],
    'ldf_series'    : ldf_series,
}

with open('ew_artifacts/early_warning_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

print('Artifacts exportés dans ew_artifacts/ :')
for p in Path('ew_artifacts').iterdir():
    print(f'  {p.name} ({p.stat().st_size // 1024 + 1} KB)')

shutil.make_archive('ew_artifacts', 'zip', 'ew_artifacts')
try:
    from google.colab import files
    files.download('ew_artifacts.zip')
    print('\new_artifacts.zip téléchargé')
except ImportError:
    print('\new_artifacts.zip prêt (exécution hors Colab)')

Artifacts exportés dans ew_artifacts/ :
  early_warning_report.json (14 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


ew_artifacts.zip téléchargé
